# SWC analysis test notebook

---
1. Step 0 - Preparing for pipeline run, creating folder, creating report, loading labels file (if exists).
2. Step 1 - Creating parquete labels file. The purpose of this file is to label each SWC neuron (by class criteria) and each synapse (post / pre).
3. Step 2 - SWC file simplification. Removing internal nodes without external leaf.
4. Step 3 - Adding labels to the simplified SWC file.
5. Step 4 - Covnerting simplified SWC file + labels to JSON format for the clumpiness calculation.
6. Step 5 - Dividing each json to multiple sub-jsons (sub-trees) of each internal node.
7. Step 6 - Clumpiness calculation via the find-clumpiness program.
8. Step 7 - Assigning clumpiness scores to the internal nodes of the SWC tree (simplified / non-simplified tree - based on node id).

In [1]:
# Import - Custom scripts
mode_list = ["og", "claude"]
mode = mode_list[1]

if mode == "og":
    from scripts.preprocessing import get_neurons_info, simplify_swc_topology, swc2json
    from scripts.processing import generate_internal_subtrees

if mode == "claude":
    from scripts.preprocessing_claude import get_neurons_info, simplify_swc_topology, swc2json
    from scripts.processing_claude import generate_internal_subtrees

from scripts.helpers import read_json

# Imports - python
import pandas as pd
import polars as pl
import numpy as np
import os

In [ ]:
# Pipeline class
class ProcessSWC():
    """
    SWC process steps:
    Step 0 - Preparing for pipeline run, creating folder, creating report, loading labels file (if exists).
    Step 1 - Creating parquete labels file. The purpose of this file is to label each SWC neuron (by class criteria) and each synapse (post / pre).
    Step 2 - SWC file simplification. Removing internal nodes without external leaf.
    Step 3 - Adding labels to the simplified SWC file.
    Step 4 - Covnerting simplified SWC file + labels to JSON format for the clumpiness calculation.
    Step 5 - Dividing each json to multiple sub-jsons (sub-trees) of each internal node.
    Step 6 - Clumpiness calculation via the find-clumpiness program.
    Step 7 - Assigning clumpiness scores to the internal nodes of the SWC tree (simplified / non-simplified tree - based on node id).
    """



    #####################################################
    ###################### STEP 0 #######################
    #####################################################
    def __init__(self, config_file: str = "config.json"):
        """
        Initiating process step of the class.
        config_file: str -> path to the `config.json` file.
        """

        config = read_json(path=config_file)
        self.dict_config = {
                            "run_name":        config["run_name"],                                   # string value of the pipeline run name
                            "swc_path":        os.path.join(*config["swc_path"].split(",")),         # absulote path of the raw swc files
                            "labels_path":     os.path.join(*config["labels_path"].split(",")),      # absulote path for the labels containing folder
                            "labels_ftr_path": os.path.join(*config["labels_ftr_path"].split(",")),  # absolute path of the labels ftr input file.
                            "overwrite_info":  bool(config["overwrite_info"]),                       # overwrite files if the `run_name` haven't been changed 
                            "data_limit":      config["data_limit"],                                 # number (int) limit of files to process
                            "n_threads":       int(config["n_threads"]),                             # number of threads to be used in the analysis
                            "n_jobs":          int(config["n_jobs"])                                 # number of jobs to process in parallel
                            } 
                         
        # Reading config for paths
        run_name = self.dict_config["run_name"]
        self.dict_paths = {
                           "0-run_log": os.path.join("output", run_name, "0-run_log"),
                           "1-output_labels": os.path.join("output", run_name, "1-output_labels"),
                           "2-swc_simplified": os.path.join("output", run_name, "2-swc_simplified"),
                           "3-swc_simplified_labeled": os.path.join("output", run_name, "3-swc_simplified_labeled"),
                           "4-json": os.path.join("output", run_name, "4-json"),
                           "5_json_divided": os.path.join("output", run_name, "5_json_divided"),
                           "6-clumpiness_scores": os.path.join("output", run_name, "6-clumpiness_scores"),
                           "7-swc_clumpiness_result": os.path.join("output", run_name, "7-swc_clumpiness_result")
                            }

        # Creating the folders defined in `self.dict_paths`
        for dir in self.dict_paths.values():
            os.makedirs(dir, exist_ok=True)



    ###############################
    ########## STEP 1 #############
    ###############################
    def step01_create_labels(self):
        """
        First step of the pipeline after the initiation, creation of a 
        parqute format labels file for each neuron in the folder.
        """

        # labels file path
        self.labels_path = os.path.join(self.dict_paths["1-output_labels"], "swc_labels.parquet")

        if os.path.exists(self.labels_path) == False:
            # Getting a list of the aviable SWC file in the swc input folder
            swc_files = [i.split(".")[0] for i in os.listdir(self.dict_config["swc_path"])]
            
            # Creating parquete file -> only relevent swc file by super-type
            get_neurons_info(parquet_path = self.dict_paths["1-output_labels"],
                             labels_path = self.dict_config["labels_path"],
                             overwrite_parquet = self.dict_config["overwrite_info"],
                             swc_labels_file_path = self.dict_config["labels_ftr_path"])



    ##############################
    ########## STEP 2 ############
    ##############################
    def step02_simplify_swc(self,
                            neuron_id: str | int):
        """
        Simplification step of the original raw SWC neuron file.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        # Loading neuron file
        neuron_path = os.path.join(self.dict_config["swc_path"], f"{neuron_id}.swc")
        neuron_swc = pd.read_csv(neuron_path, 
                                comment='#', 
                                header=None, 
                                sep=r'\s+', 
                                names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


        path_simplified_Swc = os.path.join(self.dict_paths["2-swc_simplified"], f"{neuron_id}.csv")
        if (self.dict_config["overwrite_info"]) or (os.path.exists(path_simplified_Swc) is False):
            #Simplifing SWC file
            simplify_swc_topology(swc_input = neuron_swc, 
                                swc_name = f"{neuron_id}",
                                output_path = self.dict_paths["2-swc_simplified"],
                                save_csv=True)
    
        

    ###############################
    ########### STEP 3 ############
    ###############################
    def step03_label_joining(self,
                             neuron_id: str | int):
        """
        3rd step in the pipeline - adding labels to the simplified swc file.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        # Loading relevent rows from the labels praquet file
        labels_parquet = pl.scan_parquet(self.labels_path).filter(pl.col("neuron").cast(pl.Utf8) == str(neuron_id)).collect().to_pandas()


        # Attach nodes labels to the simplified SWC files
        swc_labeled = pd.merge(left = pd.read_csv(os.path.join(self.dict_paths["2-swc_simplified"], f"{neuron_id}.csv"), index_col=0), 
                               right = labels_parquet[["node_id", "type"]].drop_duplicates(), 
                               left_on = "node_id", 
                               right_on = "node_id", 
                               how = "left")  

        #save simplified swc file
        save_path = os.path.join(self.dict_paths["3-swc_simplified_labeled"], f"{neuron_id}.csv")
        swc_labeled["type"] = swc_labeled.groupby("node_id")["type"].unique().apply(lambda X : X[0] if len(X) <= 1 else ",".join(X)) # joining labels if more then 2 per node
        swc_labeled = swc_labeled.drop_duplicates(subset=["node_id", "parent"], keep="first")                                        # dropping rows with the same parent+node_id
        
        # Saving output file
        if (self.dict_config["overwrite_info"]) or (os.path.exists(save_path) is False):
            swc_labeled.to_csv(save_path)



    ###############################
    ########### STEP 4 ############
    ###############################
    def step04_json_creation(self,
                             neuron_id:str | int):
        """
        4th step in the pipeline - creating json file from the simplified swc + labels file.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        #labels swc file import
        df_swc_labeled = pd.read_csv(os.path.join(self.dict_paths["3-swc_simplified_labeled"], f"{neuron_id}.csv"), index_col = 0)
        path_json = os.path.join(self.dict_paths["4-json"])

        # JSON conversion function
        swc2json(swc_dataset = df_swc_labeled,
                 neuron_id = neuron_id,
                 save_json = True,
                 save_path = path_json,
                 overwrite = self.dict_config["overwrite_info"])



    ###############################
    ########### STEP 5 ############
    ###############################
    def step05_internal_node_div(self,
                                 neuron_id : int | str):
        """
        5th step of the pipeline. Creating seperate JSON file for each sub-tree whithin the main tree.
        This step is useful for the individual clumpiness scoring of each internal node.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        # Required paths
        path_main_json = os.path.join(self.dict_paths["4-json"], f"{neuron_id}.json")
        path_div_json = self.dict_paths["5_json_divided"]

        # JSON split function
        generate_internal_subtrees(input_json_path = path_main_json, 
                                   neuron_number = neuron_id, 
                                   output_dir = path_div_json,
                                   overwrite = self.dict_config["overwrite_info"])



    ########################################
    ################ STEP 6 ################
    ########################################
    def step06_clumpiness_calculation(self,
                                       neuron_id : int | str):
        """
        6th step, calculating the clumpiness for each json file
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        path_main = 
        path_div = 
        paths2process = 
        pass



    ##################################
    ############# STEP 7 #############
    ##################################
    def step07_result_annealing(self):
        """
        """
        pass
        

In [3]:
# Fixed class (scripts/process_swc_claude.py) - replaces the class above when mode == "claude"
# Fixes: step 3 labels aligned by node_id, "False"/"None" config parsing, one overwrite rule for all steps,
# labels parquet path set in __init__, every step returns stats. See BUGFIX_REPORT_claude.md.
if mode == "claude":
    import importlib
    import scripts.preprocessing_claude, scripts.processing_claude, scripts.process_swc_claude

    # Reload in dependency order -> edits to the .py files are picked up without restarting the kernel
    for module in (scripts.preprocessing_claude, scripts.processing_claude, scripts.process_swc_claude):
        importlib.reload(module)

    ProcessSWC = scripts.process_swc_claude.ProcessSWC

In [4]:
# Run pipeline

if __name__ == '__main__':
    test = ProcessSWC()

In [5]:
# Test run - process a selection of neurons step by step, record failures and validate each step's output
from tqdm.auto import tqdm
import traceback
import random
import glob
import json
import time

######################
### Test settings  ###
N_NEURONS  = 5          # number of neurons to test
SELECTION  = "first"    # "first" (sorted ids) | "random"
SEED       = 0          # seed for SELECTION = "random"
NEURON_IDS = []         # explicit ids -> overrides N_NEURONS / SELECTION, e.g. ["720575940596125868"]
RUN_CHECKS = True       # validate each neuron's outputs after the steps ran
OVERWRITE  = True       # None -> config `overwrite_info`; True/False -> override for steps 2-5 only (labels parquet untouched)

# Fresh instance, so changes in the class cell are picked up (re-run the class cell first)
test = ProcessSWC()

# Step 1 - labels parquet (needed by every neuron)
test.step01_create_labels()
labels_file = os.path.join(test.dict_paths["1-output_labels"], "swc_labels.parquet")

# Steps 2-5 overwrite override (after step 1, so the 2GB labels parquet is not rebuilt)
if OVERWRITE is not None:
    test.dict_config["overwrite_info"] = OVERWRITE

# Candidate neurons -> raw SWC file AND labels exist
swc_ids = {os.path.splitext(f)[0] for f in os.listdir(test.dict_config["swc_path"]) if f.endswith(".swc")}
labeled_ids = set(pl.scan_parquet(labels_file).select(pl.col("neuron").cast(pl.Utf8)).unique().collect()["neuron"].to_list())
candidates = sorted(swc_ids & labeled_ids)

if NEURON_IDS:
    selected = [str(i) for i in NEURON_IDS]
elif SELECTION == "random":
    selected = random.Random(SEED).sample(candidates, min(N_NEURONS, len(candidates)))
else:
    selected = candidates[:N_NEURONS]

print(f"> {len(swc_ids)} SWC files, {len(candidates)} with labels, testing {len(selected)} (overwrite={test.dict_config['overwrite_info']}).")


######################
### Per-step checks ###
def check_neuron(neuron_id: str) -> dict:
    """Independent checks on the files written by steps 2-5 (does not use the pipeline functions)."""
    checks = {}

    # Step 2 - simplified tree has exactly one root
    swc_simp = pd.read_csv(os.path.join(test.dict_paths["2-swc_simplified"], f"{neuron_id}.csv"), index_col=0)
    checks["n_nodes_simplified"] = len(swc_simp)
    checks["ok_single_root"] = int((swc_simp["parent"] == -1).sum()) == 1

    # Step 3 - one row per node, and every node carries exactly its own labels
    swc_lab = pd.read_csv(os.path.join(test.dict_paths["3-swc_simplified_labeled"], f"{neuron_id}.csv"), index_col=0)
    checks["ok_unique_nodes"] = swc_lab["node_id"].is_unique

    labels = pl.scan_parquet(labels_file).filter(pl.col("neuron").cast(pl.Utf8) == neuron_id).select(["node_id", "type"]).collect().to_pandas()
    truth = labels.dropna().groupby("node_id")["type"].agg(lambda s: set(s.astype(str)))
    expected = swc_lab["node_id"].map(truth).apply(lambda v: v if isinstance(v, set) else set())
    found = swc_lab["type"].apply(lambda v: set(str(v).split(",")) if pd.notna(v) else set())
    checks["labeled_nodes_raw"] = labels["node_id"].nunique()
    checks["labeled_nodes_expected"] = int((expected.apply(len) > 0).sum())
    checks["labeled_nodes_found"] = int((found.apply(len) > 0).sum())
    checks["label_mismatches"] = int((expected != found).sum())
    checks["ok_labels"] = checks["label_mismatches"] == 0

    # Step 4 - every csv node is in the json tree, with the same labels as the step 3 csv
    with open(os.path.join(test.dict_paths["4-json"], f"{neuron_id}.json")) as f:
        tree = json.load(f)
    json_labels, stack = {}, [tree]
    while stack:
        node = stack.pop()
        json_labels[node[0]["nodeID"]] = set(node[0]["nodeLabels"])
        stack.extend(node[1])
    csv_labels = {str(int(n)): {x.strip().lower() for x in labs} for n, labs in zip(swc_lab["node_id"], found)}
    checks["ok_json_nodes"] = len(json_labels) == len(swc_lab)
    checks["ok_json_labels"] = json_labels == csv_labels

    # Step 5 - one sub-tree per internal node, root excluded
    n_internal = int(swc_lab["node_id"].isin(swc_lab["parent"]).sum())
    n_subtrees = len(glob.glob(os.path.join(test.dict_paths["5_json_divided"], neuron_id, "*.json")))
    checks["ok_subtrees"] = n_subtrees == n_internal - 1

    return checks


######################
### Run            ###
steps = [("step02", test.step02_simplify_swc),
         ("step03", test.step03_label_joining),
         ("step04", test.step04_json_creation),
         ("step05", test.step05_internal_node_div)]

records, tracebacks = [], {}
for neuron_id in tqdm(selected, desc="Testing neurons"):
    record = {"neuron": neuron_id, "status": "ok", "failed_step": None, "error": None}

    for step_name, step_fn in steps:
        t0 = time.time()
        try:
            step_fn(neuron_id)
        except Exception as err:
            record.update(status="failed", failed_step=step_name, error=f"{type(err).__name__}: {err}")
            tracebacks[neuron_id] = traceback.format_exc()
            break
        record[f"{step_name}_sec"] = round(time.time() - t0, 2)

    if RUN_CHECKS and record["status"] == "ok":
        try:
            record.update(check_neuron(neuron_id))
        except Exception as err:
            record["status"] = "check_error"
            record["error"] = f"{type(err).__name__}: {err}"
            tracebacks[neuron_id] = traceback.format_exc()

    records.append(record)

df_test = pd.DataFrame(records)


######################
### Summary        ###
print(df_test["status"].value_counts().to_string())
check_cols = [c for c in df_test.columns if c.startswith("ok_")]
if check_cols:
    print("\n> Share of neurons passing each check:")
    print(df_test[check_cols].mean().round(3).to_string())
if tracebacks:
    print(f"\n> {len(tracebacks)} neurons with errors - full traces in `tracebacks[neuron_id]`.")

df_test

c:\Users\Daniel\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


> Function execution halted, old `swc_labels.parquet` file preserved.
> 200 SWC files, 178 with labels, testing 5 (overwrite=True).


Testing neurons: 100%|██████████| 5/5 [00:43<00:00,  8.76s/it]

status
ok    5

> Share of neurons passing each check:
ok_single_root     1.0
ok_unique_nodes    1.0
ok_labels          1.0
ok_json_nodes      1.0
ok_json_labels     1.0
ok_subtrees        1.0


,neuron,status,failed_step,error,step02_sec,step03_sec,step04_sec,step05_sec,n_nodes_simplified,ok_single_root,ok_unique_nodes,labeled_nodes_raw,labeled_nodes_expected,labeled_nodes_found,label_mismatches,ok_labels,ok_json_nodes,ok_json_labels,ok_subtrees
0,720575940596125868,ok,None,None,1.55,1.16,0.07,2.07,439,True,True,274,274,274,0,True,True,True,True
1,720575940597856265,ok,None,None,0.99,0.99,0.16,7.20,1422,True,True,875,875,875,0,True,True,True,True
2,720575940597944841,ok,None,None,1.02,0.99,0.08,3.23,829,True,True,507,507,507,0,True,True,True,True
3,720575940598267657,ok,None,None,0.91,0.97,0.20,10.57,2113,True,True,1304,1304,1304,0,True,True,True,True
4,720575940599333574,ok,None,None,0.97,0.97,0.09,4.41,826,True,True,619,619,619,0,True,True,True,True
